# TrashSort Colab Training

Run the cells top to bottom.

- Upload `training_pipeline.py` and `hk_recycling_rules.json` into `/content`
- Download and prepare the dataset in `/content/taco_hk_yolo26`
- Save model outputs to Google Drive


In [ ]:
!pip install -U ultralytics opencv-python numpy requests Pillow


In [ ]:
from pathlib import Path
import json
import os
import random
import sys

from google.colab import files

DATASET_DIR = Path("/content/taco_hk_yolo26")
OUTPUT_DIR = Path("/content/drive/MyDrive/trashsort_runs")
CODE_DIR = Path("/content")

uploaded = files.upload()
required_files = ["training_pipeline.py", "hk_recycling_rules.json"]
missing = [name for name in required_files if not (CODE_DIR / name).exists()]
assert not missing, f"Missing required code files in /content: {missing}"

os.chdir(CODE_DIR)
sys.path.insert(0, str(CODE_DIR))
print(f"Code dir: {Path.cwd()}")
print(f"Dataset: {DATASET_DIR}")
print(f"Output: {OUTPUT_DIR}")


In [ ]:
from training_pipeline import (
    DEFAULT_REQUEST_TIMEOUT_S,
    DEFAULT_SEED,
    DEFAULT_VAL_RATIO,
    ExportStats,
    build_remapped_annotations,
    download_annotations,
    download_images,
    export_split,
    run_consistency_checks,
    split_ids,
    write_data_yaml,
)

DATASET_DIR.mkdir(parents=True, exist_ok=True)
raw_dir = DATASET_DIR / "raw"
raw_images_root = raw_dir / "images"

data = download_annotations(raw_dir=raw_dir, timeout_s=DEFAULT_REQUEST_TIMEOUT_S)
image_entries = list(data["images"])
downloaded_ids = download_images(
    images=image_entries,
    images_root=raw_images_root,
    timeout_s=DEFAULT_REQUEST_TIMEOUT_S,
)
labels_by_image, images_by_id, class_counter = build_remapped_annotations(data)
kept_ids = [image_id for image_id, labels in labels_by_image.items() if labels and image_id in downloaded_ids]
assert kept_ids, "No images left after remapping/filtering."
train_ids, val_ids = split_ids(ids=kept_ids, val_ratio=DEFAULT_VAL_RATIO, seed=DEFAULT_SEED)
stats = ExportStats()
rng = random.Random(DEFAULT_SEED)
export_split(
    split_name="train",
    split_ids=train_ids,
    images_by_id=images_by_id,
    labels_by_image=labels_by_image,
    raw_images_root=raw_images_root,
    output_dir=DATASET_DIR,
    stats=stats,
    rng=rng,
)
export_split(
    split_name="val",
    split_ids=val_ids,
    images_by_id=images_by_id,
    labels_by_image=labels_by_image,
    raw_images_root=raw_images_root,
    output_dir=DATASET_DIR,
    stats=stats,
    rng=rng,
)
write_data_yaml(DATASET_DIR)
(DATASET_DIR / "class_distribution.json").write_text(
    json.dumps(dict(sorted(class_counter.items(), key=lambda kv: kv[0])), indent=2),
    encoding="utf-8",
)
report = run_consistency_checks(DATASET_DIR)
assert report["ok"], report["problems"]
print(f"Prepared dataset: {DATASET_DIR}")
print(f"Train images: {stats.train_images} | Val images: {stats.val_images}")


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
print("Drive mounted.")


In [ ]:
from ultralytics import YOLO
from training_pipeline import (
    TRAIN_BATCH,
    TRAIN_BASE_MODEL,
    TRAIN_EPOCHS,
    TRAIN_IMGSZ,
    TRAIN_OPTIMIZER,
    TRAIN_PATIENCE,
    TRAIN_RUN_NAME,
    resolve_training_device,
    resolve_training_workers,
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model = YOLO(TRAIN_BASE_MODEL)
results = model.train(
    data=str(DATASET_DIR / "data.yaml"),
    epochs=TRAIN_EPOCHS,
    batch=TRAIN_BATCH,
    patience=TRAIN_PATIENCE,
    imgsz=TRAIN_IMGSZ,
    device=resolve_training_device(),
    plots=True,
    workers=resolve_training_workers(),
    optimizer=TRAIN_OPTIMIZER,
    project=str(OUTPUT_DIR),
    name=TRAIN_RUN_NAME,
    exist_ok=True,
)

print(Path(results.save_dir).resolve())
